# Parallel questions

**Difficulty:** Beginner

One document, several questions: batch them into one call, or fire one call per question — the answers come out the same either way. What changes is cost and speed, and the bigger the document, the bigger that gap.

Ask your agent if anything's unclear.

<details>
  <summary><strong>Primary source (trimmed):</strong> docs.typesafe.ai/cookbooks/parallel_questions (click to expand)</summary>

# Parallel questions

> Runs a 13-question regulatory briefing over the GDPR Wikipedia article, showing that batching every question into one TypeSafe call is 12.2x cheaper and 10.0x faster with no change in answers.

You have one document and N questions about it. You can send one request with all N questions, or N requests with one question each. With TypeSafe the answers come out the same either way: each question is scored on its own against the document, so its answer doesn't depend on what else is in the request.

To check that, the cookbook asks each question several times both ways — all N in one request, and one question per request — and compares the run-to-run std dev: how far an answer moves from one repeat to the next. Whatever noise a question has, it has under both batching strategies. Batching adds none. Most answers came back identical across all 5 repeats either way, the same value on every call, std dev exactly 0.0.

Cost and speed do change. The document dominates every request. N single-question calls pay for it N times, in N round trips; the batched call pays once. The bigger the document, the nearer that saving comes to a full Nx.

The case: a regulatory briefing over the Wikipedia article on the GDPR (~54,000 characters), checking 13 things — 8 `Noul` questions, 2 `Choice` questions, and 3 `Score` questions.

## Batching doesn't change the answers

Per question, mean and std dev of its tracked number over 5 runs, under each batching strategy:

```
question              metric             batched mean single mean  batched std  single std
breach_72h            p(yes)                    0.804       0.814       0.0055      0.0055
applies_non_eu        p(yes)                    0.990       0.990       0.0000      0.0000
dpo_all_orgs          p(yes)                    0.030       0.030       0.0000      0.0000
right_erasure         p(yes)                    0.990       0.990       0.0000      0.0000
us_federal_law        p(yes)                    0.010       0.010       0.0000      0.0000
criminal_penalties    p(yes)                    0.108       0.108       0.0045      0.0084
instrument_type       max prob                  1.000       1.000       0.0000      0.0000
individual_rights     normalized score          1.000       1.000       0.0000      0.0000
compliance_burden     normalized score          0.750       0.750       0.0000      0.0000
```

Choices, scores, and most nouls come back identical across all 5 repeats — std dev exactly 0.0 under both strategies. The two questions that carry any run-to-run noise (`breach_72h`, `criminal_penalties`) carry the *same size* of noise either way, with means agreeing within it. The noise is a property of the question, not of how you batch it: batching neither shifts the answer nor adds variance.

## The only difference: cost and speed

Same answers, different bill. The ~54,000-character article dominates every request:

```
batching                 calls        cost  total time
one call, all 13             1   $0.000497       0.27s
13 calls, one each          13   $0.006090       2.71s

batching: 12.2x cheaper, 10.0x faster
```

Cost: the 13 single-question calls re-send the article 13 times; the batched call sends it once — this saving holds regardless of how the calls are scheduled. Speed: this figure sums 13 sequential latencies; firing them concurrently would shrink the wall-clock gap, but the 13x token cost stays exactly the same either way.

> The full cookbook fetches a live, pinned Wikipedia revision and caches every call to disk — trimmed here to a short real excerpt of the same article, since the point (batch cost/speed, not answer change) doesn't need the full 54,000 characters. Read the full cookbook at [docs.typesafe.ai/cookbooks/parallel_questions](https://docs.typesafe.ai/cookbooks/parallel_questions) any time.

</details>

## The core idea

This is speculative fan-out (`lessons/patterns/0006-speculative-fan-out.ipynb`) viewed from the cost side rather than the "ask everything, ignore what you don't need" side. Every question in a request is scored independently against the same state — nothing about question *count* changes any individual answer. What scales with question count is tokens, and what scales with request count is the document you're paying to re-send.

For a short state (a sentence, a support ticket) this barely matters. For a document-dominated workload — a long article, a contract, a transcript — sending it once instead of N times is most of the savings the whole cookbook demonstrates.

In [ ]:
import { client } from "../../assets/typesafe-client.ts";
import { noul, choice, score } from "npm:@typesafe-ai/sdk";

// A real excerpt (the intro) of the same Wikipedia article the full cookbook uses,
// fetched via the same MediaWiki API - shortened here so the demo runs quickly.
const document =
  "The General Data Protection Regulation (Regulation (EU) 2016/679), abbreviated GDPR, is a " +
  "European Union regulation on information privacy in the European Union (EU) and the European " +
  "Economic Area (EEA). The GDPR is an important component of EU privacy law and human rights " +
  "law, in particular Article 8(1) of the Charter of Fundamental Rights of the European Union. " +
  "It also governs the transfer of personal data outside the EU and EEA. The GDPR's goals are to " +
  "enhance individuals' control and rights over their personal information and to simplify the " +
  "regulations for international business. It supersedes the Data Protection Directive 95/46/EC " +
  "and, among other things, simplifies the terminology.";

const QUESTIONS = {
  breach72h: noul(
    "Must a personal data breach be reported to the supervisory authority within 72 hours?",
  ),
  appliesNonEu: noul(
    "Does the regulation apply to organisations established outside the EU that offer goods or services to people in the EU?",
  ),
  instrumentType: choice("What kind of EU legal instrument is the GDPR?", {
    Regulation: "Directly binding law in all member states, no national implementation needed.",
    Directive: "Sets goals that member states implement through national law.",
    Treaty: "An international treaty between states.",
    Recommendation: "Non-binding guidance.",
  }),
  individualRights: score(
    "How strong are the rights the GDPR grants to individuals over their data?",
    [
      "None: individuals get no rights over their data.",
      "Weak: a right to be informed, but little control.",
      "Moderate: access and correction rights, but limited means to act on them.",
      "Strong: access, erasure, portability, and objection rights, with enforcement behind them.",
    ],
  ),
  complianceBurden: score(
    "How heavy is the compliance burden the GDPR places on organisations?",
    [
      "Negligible: no meaningful obligations.",
      "Light: a few notices and disclosures.",
      "Moderate: documented processes and some dedicated roles for larger processors.",
      "Heavy: records, impact assessments, officers, and breach procedures for many organisations.",
      "Extreme: obligations so demanding that ordinary organisations cannot fully comply.",
    ],
  ),
};

// One batched call: all 5 questions, document sent once
const batchedStart = performance.now();
const batchedResponse = await client.systemOne({ state: document, questions: QUESTIONS });
const batchedMs = performance.now() - batchedStart;

// 5 single-question calls: document re-sent every time
const singleStart = performance.now();
for (const [id, question] of Object.entries(QUESTIONS)) {
  await client.systemOne({ state: document, questions: { [id]: question } });
}
const singleMs = performance.now() - singleStart;

console.log("batched answers:", batchedResponse.answers);
console.log(
  `batched call: ${batchedMs.toFixed(0)}ms, ${batchedResponse.usage.input_tokens} input tokens`,
);
console.log(`5 single calls: ${singleMs.toFixed(0)}ms`);
console.log(`speedup: ${(singleMs / batchedMs).toFixed(1)}x`);

## What just happened

Five questions about the same short GDPR excerpt, asked two ways: once as a single batched call, and once as five separate calls. The *answers* should match what you'd get either way (try it — the `individualRights` score and `instrumentType` choice shouldn't depend on how many other questions came along). The *timing* is the actual point: even on a document this short, the batched call sends far fewer total tokens.

**Try it yourself:** the primary source's real saving (12.2x cheaper, 10.0x faster) came from a ~54,000-character document — over 30x longer than the one here. Predict whether swapping in a much longer piece of text as `document` would widen or narrow the speedup you measure, before trying it.

## Related

- `lessons/patterns/0006-speculative-fan-out.ipynb` — the same "ask everything in one call" idea, from the angle of not needing every answer rather than not re-paying for the document
- `cookbooks/self-consistency/nouls.ipynb`, `cookbooks/self-consistency/choices.ipynb` — also run the same question multiple times, but to measure answer stability rather than cost

Questions? Ask your agent — it has the same research this cookbook was built from, in `research/typesafe-ai.md`.